# sql_columns_exploration

In this notebook we will extract the column names from the database.
We will use this information to develop our analysis functions.
__J. A. Moreno__

First let's import some things

In [1]:
from pathlib import Path
import os
import sqlalchemy as sa
from IPython.core.debugger import set_trace

Get the current path, find out where the database is located at. Create the SQLAlchemy engine for connection

In [2]:
database_path = Path(os.path.abspath('')).absolute().parents[0] / "data/store.db"
connection_uri = "sqlite:///" + str(database_path)
engine = sa.create_engine(connection_uri)
print(engine.url)

sqlite:////home/jose/Documents/Búsqueda Laboral/BaseLabs/base-labs-liquour-analysis/data/store.db


We will load the SQL extension installed via `pip install ipython-sql`

In [3]:
%load_ext sql

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Now let's connect to the database

In [6]:
%sql engine

We can then explore the tables to get a sense of what they have for us.

In [13]:
%sqlcmd explore --table {{"SalesDec"}}

In [8]:
%sqlcmd explore --table {{"PurchasesDec"}}

In [9]:
%sqlcmd explore --table {{"VendorInvoicesDec"}}

In [10]:
%sqlcmd explore --table {{"EndInvDec"}}

In [11]:
%sqlcmd explore --table {{"BegInvDec"}}

In [12]:
%sqlcmd explore --table {{"PricingPurchasesDec"}}

## Looking for brands sold by multiple vendors
We need to verify that brands are sold by a single vendor. We can check it in both Sales and Purchases tables

In [14]:
%%sql
    -- For Sales
SELECT Brand, COUNT(DISTINCT VendorNo) AS vendor_count FROM SalesDec
    GROUP BY Brand
    HAVING vendor_count > 1
    ORDER BY vendor_count
    DESC LIMIT 20;

Running query in 'sqlite:////home/jose/Documents/Búsqueda Laboral/BaseLabs/base-labs-liquour-analysis/data/store.db'

Brand,vendor_count
90609,2
42801,2
42735,2
35977,2
31384,2
26889,2
23473,2
23472,2
21960,2
21959,2


In [15]:
%%sql
-- For Purchases
SELECT Brand, COUNT(DISTINCT VendorNumber) AS vendor_count,
    GROUP_CONCAT(DISTINCT VendorName) AS vendor_names
    FROM PurchasesDec
    GROUP BY Brand HAVING vendor_count > 1
    ORDER BY vendor_count
    DESC LIMIT 20;

Running query in 'sqlite:////home/jose/Documents/Búsqueda Laboral/BaseLabs/base-labs-liquour-analysis/data/store.db'

Brand,vendor_count,vendor_names
90609,2,"IRA GOLDMAN AND WILLIAMS, LLP ,FLAVOR ESSENCE INC"
42801,2,"MARTIGNETTI COMPANIES ,M S WALKER INC"
42735,2,"MARTIGNETTI COMPANIES ,PERFECTA WINES"
31384,2,"MARTIGNETTI COMPANIES ,M S WALKER INC"
26889,2,"FREDERICK WILDMAN & SONS ,PERFECTA WINES"
21960,2,"PINE STATE TRADING CO ,M S WALKER INC"
21959,2,"PINE STATE TRADING CO ,M S WALKER INC"
21860,2,"PINE STATE TRADING CO ,M S WALKER INC"
18771,2,"MARTIGNETTI COMPANIES,ULTRA BEVERAGE COMPANY LLP"
17754,2,"MARTIGNETTI COMPANIES ,PERFECTA WINES"


There are brands that have multiple vendors! Since the inventory tables do not preserve the vendors, it makes calculating the per-vendor COGS a bit difficult. We could just stick to purchases done in the period as an approximation for the per-vendor COGS

Let's find out how many brands are multi vendor

In [20]:
%%sql
SELECT
    COUNT(DISTINCT Brand) AS total_unique_brands,
    SUM(CASE WHEN vendor_count > 1 THEN 1 ELSE 0 END) AS total_multi_vendor
FROM (
    SELECT Brand, COUNT(DISTINCT VendorNo) AS vendor_count
    FROM SalesDec
    GROUP BY Brand
);

Running query in 'sqlite:////home/jose/Documents/Búsqueda Laboral/BaseLabs/base-labs-liquour-analysis/data/store.db'

total_unique_brands,total_multi_vendor
11237,35


In [19]:
%%sql
SELECT b.Brand, b.vendor_count, SUM(s.SalesDollars) AS revenue, SUM(s.SalesQuantity) AS units
    FROM SalesDec s JOIN 
        (SELECT Brand, COUNT(DISTINCT VendorNo) AS vendor_count
            FROM SalesDec
            GROUP BY Brand
            HAVING vendor_count > 1)
        b ON s.Brand = b.Brand
        GROUP BY b.Brand, b.vendor_count
        ORDER BY revenue DESC LIMIT 10;

Running query in 'sqlite:////home/jose/Documents/Búsqueda Laboral/BaseLabs/base-labs-liquour-analysis/data/store.db'

Brand,vendor_count,revenue,units
5299,2,761559.71,35129
5297,2,239029.83000000002,16817
5296,2,185106.59,6341
5197,2,151301.0,18000
5298,2,97668.4,5560
809,2,83735.79,4421
6692,2,78134.03,6805
2879,2,56577.98,1802
11089,2,52482.51,3977
5270,2,49291.31,35269


## Results
The overlap is minimal, we can just go ahead and use the purchases as COGS in this case.